# 03 - Stellar density maps (rubin_sim `StarMaps`)

- author : Sylvie Dagoret-Campagne
- creation date : 2026-10-01
- kernel : conda_py313_opsim53

Inspection of the stellar density maps stored in `maps/StarMaps` (copy of the `maps` data directory of [rubin_sim](https://github.com/lsst/rubin_sim), read by `rubin_sim.maf.maps.StellarDensityMap`).

Files: `starDensity_<band>_nside_64.npz` and `starDensity_<band>_wdstars_nside_64.npz` (white dwarfs only), for `<band>` in `ugrizy`. Each file contains:

* `starDensity` - array `(npix, 65)`: **cumulative** star counts per HEALPix pixel (**RING** ordering, pixel centres defined on **RA/Dec**);
  column `j` = number of stars brighter than `bins[j+1]`;
* `bins` - 66 magnitude edges, from 15.0 to 28.0 in steps of 0.2;
* `overMaxMask` - boolean array flagging the pixels exceeding the maximum density.

**Units**: the values are interpreted here as stars / deg$^2$ (rubin_sim convention); this is not stated in the files themselves, so it should be double-checked against the generating code (a pixel of NSIDE=64 covers 0.84 deg$^2$, so the two possible conventions differ only by this factor).

**Contents**
1. Inventory of the available maps
2. Loading and magnitude interpolation
3. Basic statistics and sky coverage
4. Full-sky map (equatorial and Galactic coordinates)
5. Dependence on the limiting magnitude
6. Dependence on the band
7. Views towards the Galactic centre and anticentre
8. Stellar density as a function of Galactic latitude
9. Cumulative star counts N(<m) for a few lines of sight
10. White dwarf maps

In [ ]:
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import healpy as hp
from astropy.coordinates import SkyCoord
import astropy.units as u

%matplotlib inline
plt.rcParams.update({"figure.dpi": 110, "font.size": 11})

# The notebook lives in notebooks/09_RubinMaps/ and the maps were copied to ./maps
CANDIDATES = [
    Path("maps/StarMaps"),
    Path.home() / "DATA/OpSim/maps/StarMaps",
]
DATA_DIR = next(p for p in CANDIDATES if p.is_dir())
BANDS = list("ugrizy")

## 1. Inventory of the available maps

In [ ]:
print(f"Data directory: {DATA_DIR.resolve()}\n")
for p in sorted(DATA_DIR.glob("starDensity_*.npz")):
    print(f"{p.name:40s} {p.stat().st_size / 1e6:7.2f} MB")

## 2. Loading and magnitude interpolation

`star_density(band, maglim)` returns the density of stars brighter than `maglim`, obtained by linear interpolation between the cumulative columns (0.2 mag step).

In [ ]:
_CACHE = {}


def load_star_map(band, wd=False):
    """Return (cumulative, bins, over_max_mask) for one band (wd=True: white dwarfs only)."""
    key = (band, wd)
    if key not in _CACHE:
        suffix = "_wdstars" if wd else ""
        with np.load(DATA_DIR / f"starDensity_{band}{suffix}_nside_64.npz") as f:
            _CACHE[key] = (f["starDensity"], f["bins"], f["overMaxMask"])
    return _CACHE[key]


def star_density(band, maglim, wd=False):
    """Density of stars brighter than maglim (linear interpolation in magnitude), one value per pixel."""
    cum, bins, _ = load_star_map(band, wd)
    edges = bins[1:]  # column j counts the stars brighter than edges[j]
    if maglim < edges[0]:
        return np.zeros(cum.shape[0])
    j = np.searchsorted(edges, maglim)
    if j == 0:
        return cum[:, 0].copy()
    if j >= len(edges):
        return cum[:, -1].copy()
    w = (maglim - edges[j - 1]) / (edges[j] - edges[j - 1])
    return (1 - w) * cum[:, j - 1] + w * cum[:, j]


def show_sky(
    m,
    title,
    coord=("C", "G"),
    sub=None,
    vmin=None,
    vmax=None,
    log=True,
    unit="stars / deg$^2$",
    cmap="viridis",
):
    """Mollweide view of a map in equatorial ('C') or Galactic ('G') coordinates; empty pixels in grey."""
    mm = hp.ma(np.where(m > 0, m, hp.UNSEEN))
    hp.mollview(
        mm,
        coord=list(coord),
        norm="log" if log else None,
        min=vmin,
        max=vmax,
        title=title,
        unit=unit,
        sub=sub,
        cmap=cmap,
        badcolor="lightgray",
    )
    hp.graticule(dpar=30, dmer=30, alpha=0.4)


# Pixel-centre coordinates (the maps are defined on RA/Dec)
cum_ref, bins, over_max = load_star_map("r")
NPIX = cum_ref.shape[0]
NSIDE = hp.npix2nside(NPIX)
ra, dec = hp.pix2ang(NSIDE, np.arange(NPIX), lonlat=True)
gal = SkyCoord(ra=ra * u.deg, dec=dec * u.deg, frame="icrs").galactic
GL, GB = gal.l.deg, gal.b.deg

## 3. Basic statistics and sky coverage

In [ ]:
print(
    f"NSIDE = {NSIDE}, npix = {NPIX}, pixel size = {hp.nside2resol(NSIDE, arcmin=True):.1f} arcmin, "
    f"pixel area = {hp.nside2pixarea(NSIDE, degrees=True):.3f} deg^2"
)
print(
    f"starDensity shape: {cum_ref.shape}, magnitude edges: {bins[0]:.1f} -> {bins[-1]:.1f} (step {np.diff(bins)[0]:.1f})"
)
print(f"Cumulative in magnitude for every pixel: {bool(np.all(np.diff(cum_ref, axis=1) >= 0))}")
print(f"overMaxMask: {over_max.sum()} pixels flagged\n")

empty = cum_ref[:, -1] == 0
print(f"Pixels without any star in the map (r band): {empty.sum()} / {NPIX} ({100 * empty.mean():.1f} %)")
print(f"  Dec range of these pixels: {dec[empty].min():.1f} -> {dec[empty].max():.1f} deg")

print(
    f'\n{"band":>5} {"r<20":>12} {"r<25":>12} {"r<27.8":>12}   (median over non-empty pixels, stars / deg^2)'
)
for b in BANDS:
    row = []
    for ml in (20, 25, 27.8):
        d = star_density(b, ml)
        row.append(np.median(d[d > 0]))
    print(f"{b:>5} {row[0]:12.3g} {row[1]:12.3g} {row[2]:12.3g}")

In [ ]:
# Footprint of the map: pixels with no star at the faintest limit
hp.mollview(
    empty.astype(float),
    coord=["C"],
    title="Empty pixels (1 = no star in the map), equatorial coordinates",
    cbar=False,
    cmap="Greys",
)
hp.graticule(dpar=30, dmer=30, alpha=0.4)

## 4. Full-sky map (equatorial and Galactic coordinates)

r-band density of stars brighter than r = 25 (logarithmic colour scale).

In [ ]:
MAGLIM = 25.0
d = star_density("r", MAGLIM)

fig = plt.figure(figsize=(14, 4.5))
show_sky(d, f"r < {MAGLIM}  -  equatorial (RA/Dec)", coord=["C"], sub=121, vmin=1e3, vmax=1e6)
show_sky(d, f"r < {MAGLIM}  -  Galactic (l/b)", coord=["C", "G"], sub=122, vmin=1e3, vmax=1e6)
plt.show()

## 5. Dependence on the limiting magnitude

In [ ]:
fig = plt.figure(figsize=(14, 9))
for k, ml in enumerate((20, 23, 25, 27)):
    show_sky(star_density("r", ml), f"r < {ml}", coord=["C", "G"], sub=(2, 2, k + 1), vmin=1e1, vmax=1e7)
plt.show()

## 6. Dependence on the band

Same limiting magnitude (24) in the six LSST bands.

In [ ]:
ML_BAND = 24.0
fig = plt.figure(figsize=(15, 8))
for k, b in enumerate(BANDS):
    show_sky(
        star_density(b, ML_BAND), f"{b} < {ML_BAND}", coord=["C", "G"], sub=(2, 3, k + 1), vmin=1e2, vmax=1e7
    )
plt.show()

## 7. Views towards the Galactic centre and anticentre

In [ ]:
d = star_density("r", 25.0)
mm = hp.ma(np.where(d > 0, d, hp.UNSEEN))

fig = plt.figure(figsize=(13, 5.5))
hp.gnomview(
    mm,
    rot=(0, 0),
    coord=["C", "G"],
    reso=20,
    xsize=400,
    norm="log",
    min=1e3,
    max=1e7,
    title="Galactic centre (l=0, b=0), r < 25",
    unit="stars / deg$^2$",
    sub=121,
    badcolor="lightgray",
)
hp.graticule(dpar=10, dmer=10, coord="G", alpha=0.4)
hp.gnomview(
    mm,
    rot=(180, 0),
    coord=["C", "G"],
    reso=20,
    xsize=400,
    norm="log",
    min=1e3,
    max=1e7,
    title="Anticentre (l=180, b=0), r < 25",
    unit="stars / deg$^2$",
    sub=122,
    badcolor="lightgray",
)
hp.graticule(dpar=10, dmer=10, coord="G", alpha=0.4)
plt.show()

## 8. Stellar density as a function of Galactic latitude

Median (line) and 16-84 % interval (band) over all pixels of each 5-degree latitude slice, empty pixels excluded.

In [ ]:
b_edges = np.arange(-90, 91, 5)
b_cent = 0.5 * (b_edges[1:] + b_edges[:-1])

fig, ax = plt.subplots(figsize=(8, 5))
for ml in (21, 23, 25, 27):
    d = star_density("r", ml)
    med, lo, hi = [], [], []
    for b0, b1 in zip(b_edges[:-1], b_edges[1:]):
        sel = (GB >= b0) & (GB < b1) & (d > 0)
        if sel.sum() == 0:
            med.append(np.nan)
            lo.append(np.nan)
            hi.append(np.nan)
        else:
            p16, p50, p84 = np.percentile(d[sel], [16, 50, 84])
            med.append(p50)
            lo.append(p16)
            hi.append(p84)
    (line,) = ax.plot(b_cent, med, label=f"r < {ml}")
    ax.fill_between(b_cent, lo, hi, alpha=0.2, color=line.get_color())
ax.set_yscale("log")
ax.set_xlabel("Galactic latitude b [deg]")
ax.set_ylabel("stars / deg$^2$")
ax.grid(alpha=0.3)
ax.legend()
plt.show()

## 9. Cumulative star counts N(<m) for a few lines of sight

In [ ]:
def pixel_at(l, b):
    c = SkyCoord(l=l * u.deg, b=b * u.deg, frame="galactic").icrs
    return hp.ang2pix(NSIDE, c.ra.deg, c.dec.deg, lonlat=True)


SAMPLES = {
    "Galactic centre (l=0, b=-2)": (0, -2),
    "Inner disc (l=30, b=0)": (30, 0),
    "Intermediate latitude (l=0, b=-30)": (0, -30),
    "South Galactic pole": (0, -90),
}

fig, ax = plt.subplots(figsize=(8, 5))
for name, (l, b) in SAMPLES.items():
    pix = pixel_at(l, b)
    ax.semilogy(bins[1:], cum_ref[pix], label=name)
ax.set_xlabel("r magnitude")
ax.set_ylabel("N(< r)  [stars / deg$^2$]")
ax.grid(alpha=0.3, which="both")
ax.legend()
plt.show()

## 10. White dwarf maps

`starDensity_<band>_wdstars_nside_64.npz` has the same structure as the main maps. Below: r-band white dwarf density at r < 24 and its fraction of the total stellar density.

In [ ]:
d_all = star_density("r", 24.0)
d_wd = star_density("r", 24.0, wd=True)
frac = np.where(d_all > 0, d_wd / np.where(d_all > 0, d_all, 1.0), 0.0) * 100

fig = plt.figure(figsize=(14, 4.5))
show_sky(d_wd, "White dwarfs, r < 24", coord=["C", "G"], sub=121)
show_sky(frac, "White dwarf fraction, r < 24 [%]", coord=["C", "G"], sub=122, log=False, unit="%")
plt.show()

print(
    f"Total white dwarf density (median over non-empty pixels): {np.median(d_wd[d_all > 0]):.3g} stars / deg^2"
)
print(f"Median white dwarf fraction: {np.median(frac[d_all > 0]):.2f} %")

## Notes

* The maps have NSIDE = 64 only (pixel ~ 55 arcmin), so they do not resolve structure below that scale (crowded Galactic bulge, clusters, Magellanic Clouds).
* Counts are cumulative in magnitude, tabulated every 0.2 mag; `star_density` interpolates linearly inside a bin.
* The unit of `starDensity` (stars / deg$^2$ vs stars / pixel) is assumed, not read from the files: to be confirmed with the generating code (see the `maps` README and `sims_catUtils/bin.src`).
* Next step: compare with the TRILEGAL maps in `maps/TriMaps`.